In [0]:
# ─────────────────────────────────────────────────────────────
# GOLD DIMENSION VIEWS  (presentation layer over silver)
# Views = always in sync with silver, zero extra storage/maintenance.
# Each resolves curated enrichment; PK is the *_key the fact references.
# ─────────────────────────────────────────────────────────────
CATALOG = "cricket"

# --- dim_player: fold in enrichment seed (null until you populate it) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_player AS
SELECT
    p.person_id,
    p.canonical_name,
    COALESCE(e.full_name, p.canonical_name) AS display_name,
    p.gender,
    e.country,
    e.dob,
    e.batting_style,
    e.bowling_style,
    e.bat_role,
    e.bowl_role,
    e.is_wicketkeeper,
    p.first_match_date,
    p.last_match_date
FROM {CATALOG}.silver.dim_player p
LEFT JOIN {CATALOG}.silver.dim_player_enrichment e USING (person_id)
""")

In [0]:
# --- dim_team: resolve franchise canonical (falls back to raw name) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_team AS
SELECT
    team_key,
    team_name,
    COALESCE(franchise_current_name, team_name) AS display_team,
    franchise_key,
    team_type,
    first_match_date,
    last_match_date
FROM {CATALOG}.silver.dim_team
""")

# --- dim_venue: resolve canonical venue (falls back to raw) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_venue AS
SELECT
    venue_key,
    venue_name,
    COALESCE(canonical_venue_name, venue_name) AS display_venue,
    canonical_venue_key,
    city,
    first_match_date,
    last_match_date
FROM {CATALOG}.silver.dim_venue
""")

In [0]:
# --- dim_series / dim_calendar / dim_phase: near pass-through ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_series AS
SELECT series_key, event_name, season, season_start_year,
       match_format, first_match_date, last_match_date
FROM {CATALOG}.silver.dim_series
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_calendar AS
SELECT date_key, date, year, month, day,
       month_name, day_name, quarter, iso_week, is_weekend
FROM {CATALOG}.silver.dim_calendar
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_phase AS
SELECT phase_key, phase_name, is_powerplay, powerplay_type
FROM {CATALOG}.silver.dim_phase
""")

In [0]:
# --- dim_match: gold view, with the same FK hashes the fact uses ---
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.dim_match AS
SELECT
    match_id,
    match_type, match_format, competition_variant, is_international,
    gender, season, season_start_year,
    event_name, event_match_number, event_group, event_stage,
    team_a, team_b, venue, city,
    start_date, end_date,
    CAST(date_format(start_date, 'yyyyMMdd') AS INT)            AS date_key,
    xxhash64(concat_ws('|', coalesce(event_name,'(no event)'), season)) AS series_key,
    xxhash64(lower(trim(venue)))                               AS venue_key,
    balls_per_over, scheduled_overs,
    toss_winner, toss_decision, toss_uncontested,
    outcome_winner, outcome_result, won_by_team,
    outcome_by_runs, outcome_by_wickets, outcome_by_innings,
    outcome_method, outcome_eliminator, outcome_bowl_out
FROM {CATALOG}.silver.dim_match
""")

In [0]:
CATALOG = "cricket"
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.batting_innings AS
WITH bat AS (
  SELECT
    f.batter_id AS person_id, f.match_id, f.innings_number,
    sum(f.runs_batter) AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END) AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  WHERE NOT f.is_super_over
  GROUP BY f.batter_id, f.match_id, f.innings_number
),
dism AS (
  SELECT DISTINCT w.player_out_id AS person_id, w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN {CATALOG}.silver.dim_innings i
    ON w.match_id = i.match_id AND w.innings_number = i.innings_number
  WHERE NOT i.is_super_over
    AND w.kind NOT IN ('retired hurt','retired not out')
),
innings_keys AS (
  SELECT person_id, match_id, innings_number FROM bat
  UNION
  SELECT person_id, match_id, innings_number FROM dism
)
SELECT
  ik.person_id, ik.match_id, ik.innings_number,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date,
  COALESCE(b.runs, 0)  AS runs,
  COALESCE(b.balls, 0) AS balls_faced,
  COALESCE(b.fours, 0) AS fours,
  COALESCE(b.sixes, 0) AS sixes,
  (d.person_id IS NOT NULL) AS was_out,
  (d.person_id IS NULL)     AS not_out
FROM innings_keys ik
LEFT JOIN bat  b ON ik.person_id=b.person_id AND ik.match_id=b.match_id AND ik.innings_number=b.innings_number
LEFT JOIN dism d ON ik.person_id=d.person_id AND ik.match_id=d.match_id AND ik.innings_number=d.innings_number
JOIN {CATALOG}.gold.dim_match m ON ik.match_id = m.match_id
""")
print("created batting_innings view")

In [0]:
CATALOG = "cricket"

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.bowling_innings AS
SELECT
  f.bowler_id                              AS person_id,
  f.match_id,
  f.innings_number,
  m.season,
  m.event_name,
  m.match_format,
  m.is_international,
  m.start_date,
  -- balls bowled = legal balls only (wides + no-balls excluded, they don't count to the over)
  sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END)                             AS legal_balls,
  -- runs conceded = off bat + wides + no-balls + penalty; NOT byes/leg-byes
  sum(f.runs_batter + f.extra_wides + f.extra_noballs + f.extra_penalty)       AS runs_conceded,
  -- wickets credited to the bowler (excludes run-out, retirements, obstruction)
  sum(CASE WHEN f.is_bowler_wicket THEN 1 ELSE 0 END)                          AS wickets,
  -- extras breakdown (useful for detail / economy questions)
  sum(f.extra_wides)                                                           AS wides,
  sum(f.extra_noballs)                                                         AS noballs
FROM {CATALOG}.gold.fact_ball f
JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
WHERE NOT f.is_super_over                    -- convention: super-overs excluded from figures
  AND f.bowler_id IS NOT NULL
GROUP BY
  f.bowler_id, f.match_id, f.innings_number,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date
""")

print("created bowling_innings view")

In [0]:
CATALOG = "cricket"

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.batting_innings (
  person_id       COMMENT 'Player id (stable across name spellings)',
  match_id        COMMENT 'Match identifier',
  innings_number  COMMENT 'Innings number within the match',
  season          COMMENT 'Season, e.g. 2016 or 2020/21',
  event_name      COMMENT 'Competition name, e.g. Indian Premier League',
  match_format    COMMENT 'Statistical format: T20 (incl The Hundred), ODI, Test, etc.',
  is_international COMMENT 'True for international matches, false for domestic/league',
  start_date      COMMENT 'Match start date',
  runs            COMMENT 'Runs scored by the batter in this innings',
  balls_faced     COMMENT 'Balls faced (excludes wides; a no-ball IS faced). Strike-rate denominator.',
  fours           COMMENT 'Fours hit (struck boundaries only)',
  sixes           COMMENT 'Sixes hit (struck boundaries only)',
  was_out         COMMENT 'True if dismissed this innings',
  not_out         COMMENT 'True if not out this innings'
)
COMMENT 'One row per batter per innings. All batting conventions pre-applied (super-overs excluded, correct balls-faced, retired-hurt handled, voided matches removed). Use for ALL batting stats — never compute batting from fact_ball.'
AS
WITH bat AS (
  SELECT
    f.batter_id AS person_id, f.match_id, f.innings_number,
    sum(f.runs_batter) AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END) AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  WHERE NOT f.is_super_over
  GROUP BY f.batter_id, f.match_id, f.innings_number
),
dism AS (
  SELECT DISTINCT w.player_out_id AS person_id, w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN {CATALOG}.silver.dim_innings i
    ON w.match_id = i.match_id AND w.innings_number = i.innings_number
  WHERE NOT i.is_super_over
    AND w.kind NOT IN ('retired hurt','retired not out')
),
innings_keys AS (
  SELECT person_id, match_id, innings_number FROM bat
  UNION
  SELECT person_id, match_id, innings_number FROM dism
)
SELECT
  ik.person_id, ik.match_id, ik.innings_number,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date,
  COALESCE(b.runs, 0)  AS runs,
  COALESCE(b.balls, 0) AS balls_faced,
  COALESCE(b.fours, 0) AS fours,
  COALESCE(b.sixes, 0) AS sixes,
  (d.person_id IS NOT NULL) AS was_out,
  (d.person_id IS NULL)     AS not_out
FROM innings_keys ik
LEFT JOIN bat  b ON ik.person_id=b.person_id AND ik.match_id=b.match_id AND ik.innings_number=b.innings_number
LEFT JOIN dism d ON ik.person_id=d.person_id AND ik.match_id=d.match_id AND ik.innings_number=d.innings_number
JOIN {CATALOG}.gold.dim_match m ON ik.match_id = m.match_id
""")
print("recreated batting_innings with column comments")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.bowling_innings (
  person_id       COMMENT 'Bowler id (stable across name spellings)',
  match_id        COMMENT 'Match identifier',
  innings_number  COMMENT 'Innings number within the match',
  season          COMMENT 'Season, e.g. 2016 or 2020/21',
  event_name      COMMENT 'Competition name, e.g. Indian Premier League',
  match_format    COMMENT 'Statistical format: T20 (incl The Hundred), ODI, Test',
  is_international COMMENT 'True for international, false for domestic/league',
  start_date      COMMENT 'Match start date',
  legal_balls     COMMENT 'Legal balls bowled (excludes wides and no-balls). Overs = legal_balls/6. Economy denominator.',
  runs_conceded   COMMENT 'Runs conceded: off bat + wides + no-balls + penalty. EXCLUDES byes and leg-byes.',
  wickets         COMMENT 'Wickets credited to the bowler (excludes run-outs and retirements)',
  wides           COMMENT 'Wides bowled',
  noballs         COMMENT 'No-balls bowled'
)
COMMENT 'One row per bowler per bowling innings. Bowling conventions pre-applied. Economy = 6*runs_conceded/legal_balls; average = runs_conceded/wickets; SR = legal_balls/wickets. Mat (matches played) is NOT here — comes from the squad. Use for ALL bowling stats.'
AS
SELECT
  f.bowler_id AS person_id, f.match_id, f.innings_number,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date,
  sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END) AS legal_balls,
  sum(f.runs_batter + f.extra_wides + f.extra_noballs + f.extra_penalty) AS runs_conceded,
  sum(CASE WHEN f.is_bowler_wicket THEN 1 ELSE 0 END) AS wickets,
  sum(f.extra_wides) AS wides,
  sum(f.extra_noballs) AS noballs
FROM {CATALOG}.gold.fact_ball f
JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
WHERE NOT f.is_super_over AND f.bowler_id IS NOT NULL
GROUP BY f.bowler_id, f.match_id, f.innings_number,
         m.season, m.event_name, m.match_format, m.is_international, m.start_date
""")
print("recreated bowling_innings with column comments")

In [0]:
CATALOG = "cricket"

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.player_match_summary (
  person_id    COMMENT 'Player id',
  match_id     COMMENT 'Match identifier',
  team         COMMENT 'Team the player represented in this match',
  season       COMMENT 'Season',
  event_name   COMMENT 'Competition',
  match_format COMMENT 'T20/ODI/Test etc.',
  is_international COMMENT 'True for internationals',
  start_date   COMMENT 'Match date',
  team_won     COMMENT 'True if the player''s team won this match',
  batted       COMMENT 'True if the player batted (had a batting innings)',
  runs         COMMENT 'Runs scored (0 if did not bat)',
  balls_faced  COMMENT 'Balls faced batting',
  was_out      COMMENT 'True if dismissed while batting',
  bowled       COMMENT 'True if the player bowled',
  wickets      COMMENT 'Wickets taken (0 if did not bowl)',
  runs_conceded COMMENT 'Runs conceded bowling',
  legal_balls  COMMENT 'Legal balls bowled'
)
COMMENT 'One row per player per match they were in the squad. Authoritative source for Matches Played (Mat) — counts squad appearances whether or not they batted/bowled. Carries team_won for outcome/impact analysis. Voided matches excluded.'
AS
WITH squad AS (
  -- base grain: everyone in the XI (excludes voided matches)
  SELECT DISTINCT mp.person_id, mp.match_id, mp.team
  FROM {CATALOG}.silver.match_player mp
  WHERE mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
),
bat AS (
  -- batting line per player-match (sum across innings, though usually one in limited-overs)
  SELECT person_id, match_id,
         sum(runs) AS runs, sum(balls_faced) AS balls_faced,
         max(CASE WHEN was_out THEN 1 ELSE 0 END) = 1 AS was_out,
         true AS batted
  FROM {CATALOG}.gold.batting_innings
  GROUP BY person_id, match_id
),
bowl AS (
  SELECT person_id, match_id,
         sum(wickets) AS wickets, sum(runs_conceded) AS runs_conceded,
         sum(legal_balls) AS legal_balls,
         true AS bowled
  FROM {CATALOG}.gold.bowling_innings
  GROUP BY person_id, match_id
)
SELECT
  s.person_id, s.match_id, s.team,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date,
  (m.won_by_team = s.team)                       AS team_won,
  COALESCE(b.batted, false)                      AS batted,
  COALESCE(b.runs, 0)                            AS runs,
  COALESCE(b.balls_faced, 0)                     AS balls_faced,
  COALESCE(b.was_out, false)                     AS was_out,
  COALESCE(bw.bowled, false)                     AS bowled,
  COALESCE(bw.wickets, 0)                        AS wickets,
  COALESCE(bw.runs_conceded, 0)                  AS runs_conceded,
  COALESCE(bw.legal_balls, 0)                    AS legal_balls
FROM squad s
JOIN {CATALOG}.gold.dim_match m ON s.match_id = m.match_id
LEFT JOIN bat  b  ON s.person_id = b.person_id  AND s.match_id = b.match_id
LEFT JOIN bowl bw ON s.person_id = bw.person_id AND s.match_id = bw.match_id
""")
print("created player_match_summary view")

In [0]:
CATALOG = "cricket"

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.player_match_summary (
  person_id       COMMENT 'Player id (stable across name spellings)',
  match_id        COMMENT 'Match identifier',
  team            COMMENT 'Team the player represented in this match',
  season          COMMENT 'Season, e.g. 2016 or 2020/21',
  event_name      COMMENT 'Competition name, e.g. Indian Premier League',
  match_format    COMMENT 'Statistical format: T20 (incl The Hundred), ODI, Test',
  is_international COMMENT 'True for international matches, false for domestic/league',
  start_date      COMMENT 'Match start date',
  team_won        COMMENT 'True if the players team won this match. Use for win %, impact, and outcome questions.',
  batted          COMMENT 'True if the player batted in this match',
  runs            COMMENT 'Runs scored by the player (0 if did not bat)',
  balls_faced     COMMENT 'Balls faced batting (for match strike rate = 100*runs/balls_faced)',
  was_out         COMMENT 'True if dismissed while batting',
  bowled          COMMENT 'True if the player bowled in this match',
  wickets         COMMENT 'Wickets taken (0 if did not bowl)',
  runs_conceded   COMMENT 'Runs conceded bowling',
  legal_balls     COMMENT 'Legal balls bowled'
)
COMMENT 'One row per player per match they were in the squad. AUTHORITATIVE source for Matches Played (Mat) — counts squad appearances whether or not the player batted/bowled. Carries team_won for win %, impact, and outcome analysis. Voided matches excluded. Use this (not batting_innings/bowling_innings) for match counts and any team-result question.'
AS
WITH squad AS (
  SELECT DISTINCT mp.person_id, mp.match_id, mp.team
  FROM {CATALOG}.silver.match_player mp
  WHERE mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
),
bat AS (
  SELECT person_id, match_id,
         sum(runs) AS runs, sum(balls_faced) AS balls_faced,
         max(CASE WHEN was_out THEN 1 ELSE 0 END) = 1 AS was_out,
         true AS batted
  FROM {CATALOG}.gold.batting_innings
  GROUP BY person_id, match_id
),
bowl AS (
  SELECT person_id, match_id,
         sum(wickets) AS wickets, sum(runs_conceded) AS runs_conceded,
         sum(legal_balls) AS legal_balls,
         true AS bowled
  FROM {CATALOG}.gold.bowling_innings
  GROUP BY person_id, match_id
)
SELECT
  s.person_id, s.match_id, s.team,
  m.season, m.event_name, m.match_format, m.is_international, m.start_date,
  (m.won_by_team = s.team)                       AS team_won,
  COALESCE(b.batted, false)                      AS batted,
  COALESCE(b.runs, 0)                            AS runs,
  COALESCE(b.balls_faced, 0)                     AS balls_faced,
  COALESCE(b.was_out, false)                     AS was_out,
  COALESCE(bw.bowled, false)                     AS bowled,
  COALESCE(bw.wickets, 0)                        AS wickets,
  COALESCE(bw.runs_conceded, 0)                  AS runs_conceded,
  COALESCE(bw.legal_balls, 0)                    AS legal_balls
FROM squad s
JOIN {CATALOG}.gold.dim_match m ON s.match_id = m.match_id
LEFT JOIN bat  b  ON s.person_id = b.person_id  AND s.match_id = b.match_id
LEFT JOIN bowl bw ON s.person_id = bw.person_id AND s.match_id = bw.match_id
""")
print("recreated player_match_summary with column comments")

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.gold.player_match_summary").show(20, truncate=False)

In [0]:
#spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.silver.dim_player_enrichment (
#    person_id       STRING,
#    full_name       STRING,
#    country         STRING,
#    dob             DATE,
#    batting_style   STRING,
#    bowling_style   STRING,
#    bat_role        STRING,
#    bowl_role       STRING,
#    is_wicketkeeper BOOLEAN,
#    source          STRING,
#    enriched_at     TIMESTAMP
#)""")